In [ ]:
import os
from dotenv import load_dotenv

In [ ]:
load_dotenv()

OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY")

In [ ]:
from dotenv import load_dotenv
from langchain.chat_models import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage, BaseMessage, AIMessage
from langgraph.graph import StateGraph, START, END
from typing import List, Literal, TypedDict
from pydantic import BaseModel, Field


load_dotenv()

llm_gpt4omini_research = ChatOpenAI(
    base_url="https://openrouter.ai/api/v1",
    openai_api_key=OPENROUTER_API_KEY,
    model_name="gpt-4o-mini",
    temperature=0.3
)


llm_aldr_work = ChatOpenAI(
    base_url="https://openrouter.ai/api/v1",
    openai_api_key=OPENROUTER_API_KEY,
    model_name="alibaba/tongyi-deepresearch-30b-a3b:free",
    temperature=0.1
)


llm_nsllama_talk = ChatOpenAI(
    base_url="https://openrouter.ai/api/v1",
    openai_api_key=OPENROUTER_API_KEY,
    model_name="nousresearch/deephermes-3-llama-3-8b-preview:free",
    temperature=0.7
)

In [ ]:
class TaskClassification(BaseModel):
    task_type: Literal["code", "dialog", "local"] = Field(
        description="Тип задачи: code - программирование, dialog - общение, local - российские реалии"
    )
    confidence: float = Field(
        description="Уверенность в классификации от 0.0 до 1.0",
        ge=0.0, le=1.0
    )
    reasoning: str = Field(
        description="Краткое объяснение выбора",
        max_length=100
    )


class MultiModelState(TypedDict):
    user_question: str          # Вопрос пользователя
    task_type: str              # Результат классификации
    code_analysis: str          # Результат от alibaba
    dialog_response: str        # Результат от llama
    local_context: str          # Результат от gpt
    final_answer: str           # Итоговый ответ
    should_continue: bool       # Продолжать работу